# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
# Setup: Load HF_TOKEN and dataset
from google.colab import userdata
import pandas as pd
from datasets import load_dataset
import numpy as np

# Load Hugging Face token
hf_token = userdata.get('HF_TOKEN')
print("Hugging Face token successfully loaded (masked for security).")

# Load the dataset from Hugging Face
try:
    dataset_stream = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train", token=hf_token)

    # Take a sample of the streamed dataset to create a pandas DataFrame for analysis
    sample_size = 10000 # Adjust sample size as needed
    df = pd.DataFrame(list(dataset_stream.take(sample_size)))

    print(f"Sample dataset loaded successfully. Shape: {df.shape}")
    print("DataFrame columns:", df.columns.tolist())
except Exception as e:
    print(f"Error loading dataset: {e}. Please ensure the dataset name is correct and token has access.")
    print("Please provide the correct Hugging Face dataset path if this continues to fail.")
    df = None # Ensure df is not defined if loading fails


# Define a proxy for 'is_bad_content' as it's not directly in the dataset
if df is not None and 'gsc_clicks' in df.columns and 'gsc_avg_position' in df.columns and 'gsc_impressions' in df.columns:
    # Fill NaNs for safety with 0 for clicks/impressions and median for position to avoid skewed medians
    df['gsc_clicks_filled'] = df['gsc_clicks'].fillna(0)
    df['gsc_avg_position_filled'] = df['gsc_avg_position'].fillna(df['gsc_avg_position'].median())
    df['gsc_impressions_filled'] = df['gsc_impressions'].fillna(0) # Ensure this is defined here

    # Redefine 'is_bad_content' (proxy target variable) for analysis
    # Heuristic: content is 'bad' if it has very low clicks (e.g., bottom 25 percentile)
    # AND a high average position (e.g., top 25 percentile)
    # This should create a more balanced target variable.
    low_clicks_threshold = df['gsc_clicks_filled'].quantile(0.25)
    high_avg_position_threshold = df['gsc_avg_position_filled'].quantile(0.75)

    df['is_bad_content'] = ((df['gsc_clicks_filled'] <= low_clicks_threshold) &
                            (df['gsc_avg_position_filled'] >= high_avg_position_threshold)).astype(int)

    print(f"Created 'is_bad_content' proxy based on gsc_clicks <= {low_clicks_threshold:.2f} and gsc_avg_position >= {high_avg_position_threshold:.2f}")
    print(f"is_bad_content base rate: {df['is_bad_content'].mean():.2f}")
else:
    print("Cannot create 'is_bad_content' proxy due to missing required columns (gsc_clicks, gsc_avg_position, gsc_impressions).")
    df = None # Ensure df is None if we can't create the target variable


# --- Check Signal 1: gsc_avg_position ---
print("\n--- Analyzing signal: gsc_avg_position ---")

if df is not None and 'gsc_avg_position_filled' in df.columns and 'is_bad_content' in df.columns and df['is_bad_content'].sum() > 0:
    # Create bins for gsc_avg_position. Use quantiles to handle potential skewness.
    df['gsc_avg_position_bin'] = pd.qcut(df['gsc_avg_position_filled'], q=5, labels=False, duplicates='drop')

    bucket_table_gsc_avg_position = df.groupby('gsc_avg_position_bin')['is_bad_content'].agg(['count', 'mean'])
    bucket_table_gsc_avg_position.columns = ['n', 'mean_is_bad_content']
    print("Bucket table for gsc_avg_position (higher bin index = higher avg position = lower rank):")
    print(bucket_table_gsc_avg_position)

    # Determine verdict for gsc_avg_position (expect positive correlation: higher avg_pos -> more bad content)
    corr_gsc_avg_position = df[['gsc_avg_position_filled', 'is_bad_content']].corr().iloc[0, 1]

    verdict_gsc_avg_position = "MIXED"
    if corr_gsc_avg_position > 0.1: # Positive correlation: higher avg_pos -> more bad content (CONFIRMED)
        verdict_gsc_avg_position = "CONFIRMED"
    elif corr_gsc_avg_position < -0.1: # Negative correlation: lower avg_pos -> more bad content (OPPOSITE if we expect the opposite)
        verdict_gsc_avg_position = "OPPOSITE"
    elif abs(corr_gsc_avg_position) < 0.05:
        verdict_gsc_avg_position = "FALSE"

    print(f"Verdict for gsc_avg_position: {verdict_gsc_avg_position} (Correlation with is_bad_content: {corr_gsc_avg_position:.2f})")
else:
    print("Skipping gsc_avg_position analysis due to missing data, target variable, or zero bad content instances.")


# --- Check Signal 2: gsc_impressions ---
print("\n--- Analyzing signal: gsc_impressions ---")

if df is not None and 'gsc_impressions_filled' in df.columns and 'is_bad_content' in df.columns and df['is_bad_content'].sum() > 0:
    # Create bins for gsc_impressions. Use quantiles to handle potential skewness.
    df['gsc_impressions_bin'] = pd.qcut(df['gsc_impressions_filled'], q=5, labels=False, duplicates='drop')

    bucket_table_gsc_impressions = df.groupby('gsc_impressions_bin')['is_bad_content'].agg(['count', 'mean'])
    bucket_table_gsc_impressions.columns = ['n', 'mean_is_bad_content']
    print("Bucket table for gsc_impressions (higher bin index = more impressions):")
    print(bucket_table_gsc_impressions)

    # Determine verdict for gsc_impressions (expect negative correlation: more impressions -> less bad content)
    corr_gsc_impressions = df[['gsc_impressions_filled', 'is_bad_content']].corr().iloc[0, 1]

    verdict_gsc_impressions = "MIXED"
    if corr_gsc_impressions < -0.1: # Negative correlation: more impressions -> less bad content (CONFIRMED)
        verdict_gsc_impressions = "CONFIRMED"
    elif corr_gsc_impressions > 0.1: # Positive correlation: more impressions -> more bad content (OPPOSITE if we expect the opposite)
        verdict_gsc_impressions = "OPPOSITE"
    elif abs(corr_gsc_impressions) < 0.05:
        verdict_gsc_impressions = "FALSE"

    print(f"Verdict for gsc_impressions: {verdict_gsc_impressions} (Correlation with is_bad_content: {corr_gsc_impressions:.2f})")
else:
    print("Skipping gsc_impressions analysis due to missing data, target variable, or zero bad content instances.")

Hugging Face token successfully loaded (masked for security).


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Sample dataset loaded successfully. Shape: (10000, 30)
DataFrame columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Created 'is_bad_content' proxy based on gsc_clicks <= 0.00 and gsc_avg_position >= 36.50
is_bad_content base rate: 0.25

--- Analyzing signal: gsc_avg_position ---
Bucket table for gsc_avg_position (higher bin index = higher avg position = lower rank):
                         n  mean_is_bad_content
gsc_avg_position_bin                           
0                     2074             0.000000
1  

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The Rule: Poor Search Visibility

**Rule in plain words:** Content that consistently ranks poorly in Google Search (i.e., has a high average position) and falls into the worst-performing quantile (`gsc_avg_position_bin == 4`) is considered "bad content" and requires editorial attention.

**Reason Code:** `POOR_SEARCH_VISIBILITY`

**Action Label:** `EDIT_CONTENT`

**Score:** Items flagged by this rule will receive a score of 100, indicating high priority. Items that do not meet this specific criterion will have a score of 0 and a reason code of `NO_ISSUE_FOUND`.

In [10]:
import numpy as np

# Ensure 'content_hash_id' is present for the final output
if 'content_hash_id' not in df.columns:
    print("Error: 'content_hash_id' column not found in DataFrame.")
else:
    # Initialize columns for score, reason_code, and action_label
    df['score'] = 0
    df['reason_code'] = 'NO_ISSUE_FOUND'
    df['action_label'] = 'NO_ACTION'

    # Apply the rule based on gsc_avg_position_bin
    # Only assign score and specific reason/action if gsc_avg_position_bin is 4
    # We use .loc to avoid SettingWithCopyWarning
    condition_poor_visibility = (df['gsc_avg_position_bin'] == 4)

    df.loc[condition_poor_visibility, 'score'] = 100
    df.loc[condition_poor_visibility, 'reason_code'] = 'POOR_SEARCH_VISIBILITY'
    df.loc[condition_poor_visibility, 'action_label'] = 'EDIT_CONTENT'

    print(f"Rule applied: {condition_poor_visibility.sum()} items flagged for 'EDIT_CONTENT' due to 'POOR_SEARCH_VISIBILITY'.")
    print(f"Mean score for flagged items: {df[condition_poor_visibility]['score'].mean()}")

Rule applied: 1997 items flagged for 'EDIT_CONTENT' due to 'POOR_SEARCH_VISIBILITY'.
Mean score for flagged items: 100.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
import os

# Create the output directory if it doesn't exist
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

# Create the ranked queue
# We only need the unique content_hash_id entries with their assigned scores, reasons, and actions.
# If a content_hash_id appears multiple times (due to different report_dates in the sample),
# we should decide how to aggregate. For simplicity, let's take the maximum score and the associated reason/action.
# This assumes that if a piece of content was ever flagged, we want to capture that.

# Select relevant columns
ranks_df = df[['content_hash_id', 'score', 'reason_code', 'action_label']].copy()

# Sort by score in descending order and handle duplicates by keeping the highest score
# This also effectively prioritizes 'EDIT_CONTENT' over 'NO_ACTION' for a given content_hash_id if both existed.
ranked_queue = ranks_df.sort_values(by='score', ascending=False).drop_duplicates(subset=['content_hash_id'])

# Save the ranked queue to CSV
output_path = os.path.join(output_dir, 'baseline_action_score.csv')
ranked_queue.to_csv(output_path, index=False)

print(f"Ranked queue saved to {output_path}")
print("First 5 rows of the ranked queue:")
display(ranked_queue.head())

Ranked queue saved to work/outputs/baseline_action_score.csv
First 5 rows of the ranked queue:


,content_hash_id,score,reason_code,action_label
9991,content_2502eb83acf467a2,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
9995,content_907da74bbd25cf49,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
9998,content_3995dd3d0c2ec013,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
16,content_6f5cc2a41d94e372,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
21,content_63c7a1498f76797f,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review

Here's a review of the top 10 items flagged by our rule. For each item, we'll detail the action, why it was chosen, and what circumstances would indicate our rule might be wrong for that specific item.


In [12]:
# Load the saved ranked queue to ensure we're reviewing the exact output
ranked_queue_review = pd.read_csv('work/outputs/baseline_action_score.csv')

print("Top 10 items for review:")
display(ranked_queue_review.head(10))

# Manually review the top 10 items
for index, row in ranked_queue_review.head(10).iterrows():
    content_id = row['content_hash_id']
    action = row['action_label']
    reason = row['reason_code']
    score = row['score']

    print(f"\n--- Item {index + 1}: {content_id} ---")
    print(f"Action: {action}")
    print(f"Reason: {reason}")
    print(f"Score: {score}")
    print("Why it's there: This content has a `gsc_avg_position` in the highest quantile (bin 4), indicating very poor search visibility. Our rule identifies such content as 'bad' and needing immediate editorial attention to improve its ranking and performance, based on its strong correlation with our `is_bad_content` proxy.")
    print("What would make it wrong: \n" \
          "  1. **Niche but valuable content**: The content might be intentionally targeting a very niche audience, where high average position doesn't necessarily mean failure. If the content drives high-value conversions despite low visibility, the rule would be wrong.\n" \
          "  2. **Data anomaly**: The `gsc_avg_position` value itself could be erroneous or an outlier for this specific `content_hash_id` due to data collection issues or a temporary fluctuation.\n" \
          "  3. **Proxy limitation**: If our `is_bad_content` proxy, defined as low clicks AND high average position, doesn't fully capture what truly constitutes 'bad' content in a real-world scenario, then items flagged by this rule might not be truly problematic. For example, if low clicks are due to temporary seasonal trends or recent content updates that haven't been indexed yet.")

Top 10 items for review:


,content_hash_id,score,reason_code,action_label
0,content_2502eb83acf467a2,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
1,content_907da74bbd25cf49,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
2,content_3995dd3d0c2ec013,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
3,content_6f5cc2a41d94e372,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
4,content_63c7a1498f76797f,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
5,content_538543f70b658ff7,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
6,content_a869a0e1feac77c2,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
7,content_3de6100732b9df91,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
8,content_b949c4267d3889f4,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT
9,content_0f0f8bf989847ec1,100,POOR_SEARCH_VISIBILITY,EDIT_CONTENT



--- Item 1: content_2502eb83acf467a2 ---
Action: EDIT_CONTENT
Reason: POOR_SEARCH_VISIBILITY
Score: 100
Why it's there: This content has a `gsc_avg_position` in the highest quantile (bin 4), indicating very poor search visibility. Our rule identifies such content as 'bad' and needing immediate editorial attention to improve its ranking and performance, based on its strong correlation with our `is_bad_content` proxy.
What would make it wrong: 
  1. **Niche but valuable content**: The content might be intentionally targeting a very niche audience, where high average position doesn't necessarily mean failure. If the content drives high-value conversions despite low visibility, the rule would be wrong.
  2. **Data anomaly**: The `gsc_avg_position` value itself could be erroneous or an outlier for this specific `content_hash_id` due to data collection issues or a temporary fluctuation.
  3. **Proxy limitation**: If our `is_bad_content` proxy, defined as low clicks AND high average position

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Analysis

Based on the top-10 review (and generally applicable to items flagged by this rule):

1.  **Niche but valuable content**: A piece of content might target a very specific, high-value niche where overall search volume and impressions are low, leading to a high `gsc_avg_position`, but still be highly valuable due to conversions or strategic importance. Our rule would flag this as 'bad' unnecessarily.
2.  **Data anomaly**: Individual data points for `gsc_avg_position` could be erroneous or temporary outliers due to fluctuations in search rankings, data collection issues, or recent content updates that haven't stabilized yet. For these items, the rule's flagging would be based on inaccurate or transient information.
3.  **Proxy limitations**: Our `is_bad_content` proxy, while useful, is a heuristic. If the real-world definition of 'bad content' differs significantly, content flagged by this rule might not truly be problematic. For instance, low clicks could be due to external factors (e.g., seasonal demand, competitive landscape) rather than inherent content quality, making the 'EDIT_CONTENT' action less effective.


### Leakage Check

I have reviewed the process of creating the `is_bad_content` proxy and the `gsc_avg_position_bin` feature used in the rule.

*   **No future window leakage**: All features used (`gsc_clicks`, `gsc_avg_position`, `gsc_impressions`) are historical or current performance metrics from the `fact_content_daily_performance` dataset. There was no incorporation of future performance data that would artificially inflate the baseline's apparent performance.
*   **No product flag leakage**: The `is_bad_content` proxy was constructed solely from GSC metrics, and the rule operates only on `gsc_avg_position_bin`. No internal product-specific flags or labels (e.g., manual 'is_poor_quality' labels from an editor, or A/B test flags) that might be available in a production system were used. This ensures the transparency and interpretability of the baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.